# Module 3: Agent Architecture

This module explains **how an AI agent is structured internally** and how it decides, acts, observes results, and continues toward a goal.

---

# 1. Sense–Think–Act Loop

The most basic agent architecture is:



```text
Sense → Think → Act
```



A more complete version is:



```text
Observe → Reason → Act → Observe → Repeat
```



## Sense or Observe

The agent receives information from the environment.

Examples:

- user message,
- API response,
- database result,
- file content,
- tool error,
- another agent’s output.

## Think or Reason

The agent decides:

- what the user wants,
- what information is missing,
- which tool should be called,
- whether the task is complete,
- whether it should retry or re-plan.

## Act

The agent performs an action.

Examples:

- call an API,
- query a database,
- search documents,
- send an email,
- ask the user a question,
- return a final answer.

---

# 2. Basic Agent Loop



```text
User goal
   ↓
Read current state
   ↓
Choose next action
   ↓
Execute tool
   ↓
Observe tool result
   ↓
Update state
   ↓
Goal completed?
   ├── Yes → Final response
   └── No  → Continue
```



This loop is the foundation of most AI-agent systems.

---

# 3. ReAct Architecture

**ReAct** means:



```text
Reason + Act
```



The agent alternates between reasoning and action.

Conceptually:



```text
Thought
  ↓
Action
  ↓
Observation
  ↓
Thought
  ↓
Action
  ↓
Observation
  ↓
Final answer
```



## Example

User:

> What is the current weather in Delhi, and should I carry an umbrella?

The agent may proceed like this:



```text
Reason:
I need current weather information.

Action:
Call get_weather("Delhi")

Observation:
Rain probability is 80%.

Reason:
The rain probability is high, so an umbrella is recommended.

Final answer:
Yes, carry an umbrella.
```



The internal reasoning may not always be exposed to the user, but the architecture still follows the same loop.

---

# 4. ReAct in LangGraph

A basic ReAct-style graph may look like:



```text
START
  ↓
agent_node
  ↓
Should call a tool?
  ├── Yes → tool_node
  │          ↓
  │       agent_node
  │
  └── No → END
```



The agent node decides whether to:

- call a tool,
- continue reasoning,
- or return the final answer.

---

## Basic LangGraph Structure



In [ ]:
from typing_extensions import TypedDict, Annotated
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages


class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]




The state contains the conversation and tool-call history.

---

## Agent Node



In [ ]:
def agent_node(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])

    return {
        "messages": [response]
    }




The model reads all messages and decides whether it needs a tool.

---

## Routing Function



In [ ]:
from typing import Literal


def should_continue(
    state: AgentState
) -> Literal["tools", "__end__"]:

    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"

    return "__end__"




This function checks the model’s latest response.

If the model generated a tool call:



```text
agent → tools
```



Otherwise:



```text
agent → END
```



---

# 5. Tool Node

A tool node executes the requested tool.

For example:



In [ ]:
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """Return weather information for a city."""
    return f"The weather in {city} is rainy."




Bind the tool to the LLM:



In [ ]:
llm_with_tools = llm.bind_tools(
    [get_weather]
)




Create a tool node:



In [ ]:
from langgraph.prebuilt import ToolNode


tool_node = ToolNode(
    [get_weather]
)




The tool node:

1. Reads the model’s tool call.
2. Executes the matching function.
3. Converts the result into a tool message.
4. Adds the result back into state.

---

# 6. Complete ReAct Graph Example



In [ ]:
from typing import Literal
from typing_extensions import TypedDict, Annotated

from langchain_core.messages import AnyMessage, HumanMessage
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode


# ==========================================================
# 1. Define Tool
# ==========================================================

@tool
def get_weather(city: str) -> str:
    """Get weather information for a city."""

    weather_data = {
        "Delhi": "Rainy, with an 80% chance of rain",
        "Mumbai": "Cloudy",
        "Jaipur": "Sunny"
    }

    return weather_data.get(
        city,
        "Weather information is unavailable"
    )


# ==========================================================
# 2. Bind Tool to LLM
# ==========================================================

llm_with_tools = llm.bind_tools(
    [get_weather]
)


# ==========================================================
# 3. Define State
# ==========================================================

class AgentState(TypedDict):
    messages: Annotated[
        list[AnyMessage],
        add_messages
    ]


# ==========================================================
# 4. Define Agent Node
# ==========================================================

def agent_node(state: AgentState):
    response = llm_with_tools.invoke(
        state["messages"]
    )

    return {
        "messages": [response]
    }


# ==========================================================
# 5. Define Routing Logic
# ==========================================================

def should_continue(
    state: AgentState
) -> Literal["tools", "__end__"]:

    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"

    return "__end__"


# ==========================================================
# 6. Create Tool Node
# ==========================================================

tool_node = ToolNode(
    [get_weather]
)


# ==========================================================
# 7. Build Graph
# ==========================================================

builder = StateGraph(AgentState)

builder.add_node(
    "agent",
    agent_node
)

builder.add_node(
    "tools",
    tool_node
)

builder.add_edge(
    START,
    "agent"
)

builder.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "__end__": END
    }
)

builder.add_edge(
    "tools",
    "agent"
)

graph = builder.compile()


# ==========================================================
# 8. Invoke Graph
# ==========================================================

result = graph.invoke({
    "messages": [
        HumanMessage(
            content=(
                "What is the weather in Delhi, "
                "and should I carry an umbrella?"
            )
        )
    ]
})

print(
    result["messages"][-1].content
)




---

# How This Graph Works

## Step 1: User message enters state



In [ ]:
{
    "messages": [
        HumanMessage(
            content="What is the weather in Delhi?"
        )
    ]
}




## Step 2: Agent node runs

The model reads the message and generates a tool call:



```text
get_weather(city="Delhi")
```



State now contains:



```text
HumanMessage
AIMessage containing tool call
```



## Step 3: Router checks the last message

Because `tool_calls` exists:



```text
agent → tools
```



## Step 4: Tool node executes the function



In [ ]:
get_weather("Delhi")




Tool result:



```text
Rainy, with an 80% chance of rain
```



This result is added as a tool message.

## Step 5: Graph returns to the agent node

The model now sees:



```text
User question
Tool call
Tool result
```



It generates the final answer:



```text
The weather in Delhi is rainy with an 80% chance of rain,
so you should carry an umbrella.
```



## Step 6: Router runs again

This time the model did not request another tool.

Therefore:



```text
agent → END
```



---

# 7. Plan-and-Execute Architecture

ReAct selects one action at a time.

**Plan-and-execute** first creates a broader plan and then executes the steps.



```text
User goal
   ↓
Planner
   ↓
Task list
   ↓
Executor
   ↓
Execute one task
   ↓
Update progress
   ↓
More tasks?
   ├── Yes → Continue
   └── No  → Final answer
```



## Example

User:

> Prepare a competitor analysis report.

Planner creates:



```text
1. Identify competitors.
2. Collect product information.
3. Compare pricing.
4. Compare features.
5. Summarize strengths and weaknesses.
6. Prepare the report.
```



The executor then performs each task.

---

# ReAct vs Plan-and-Execute

| ReAct | Plan-and-Execute |
|---|---|
| Chooses one step at a time | Creates a plan first |
| Flexible | More structured |
| Good for short tasks | Good for complex tasks |
| Easy to adapt | Easier to track progress |
| May lose long-term direction | May require re-planning |

---

# 8. Planner Node Example

A LangGraph state for planning may look like:



In [ ]:
class PlanningState(TypedDict, total=False):
    user_goal: str
    plan: list[str]
    current_step: int
    completed_steps: list[str]
    final_answer: str




Planner node:



In [ ]:
def planner_node(
    state: PlanningState
) -> dict:

    plan = [
        "Search competitors",
        "Collect pricing information",
        "Compare features",
        "Generate report"
    ]

    return {
        "plan": plan,
        "current_step": 0,
        "completed_steps": []
    }




Executor node:



In [ ]:
def executor_node(
    state: PlanningState
) -> dict:

    current_step = state["current_step"]
    task = state["plan"][current_step]

    result = f"Completed: {task}"

    return {
        "completed_steps": (
            state["completed_steps"] + [result]
        ),
        "current_step": current_step + 1
    }




Router:



In [ ]:
from typing import Literal


def route_execution(
    state: PlanningState
) -> Literal["executor", "finalize"]:

    if state["current_step"] < len(state["plan"]):
        return "executor"

    return "finalize"




---

# 9. Reflection Architecture

Reflection means that the agent evaluates its own result before finalizing it.



```text
Generate answer
      ↓
Critique answer
      ↓
Is it good enough?
   ├── Yes → Final answer
   └── No  → Improve answer
```



## Example

The agent writes a report.

A critic node checks:

- Are important facts missing?
- Is the answer logically correct?
- Are sources included?
- Does it satisfy the user’s request?
- Is any claim unsupported?

If the answer is weak, the graph sends it back for revision.

---

# Reflection State Example



In [ ]:
class ReflectionState(TypedDict, total=False):
    user_query: str
    draft_answer: str
    critique: str
    revision_count: int
    final_answer: str




Generate node:



In [ ]:
def generate_node(state):
    return {
        "draft_answer": "Initial answer...",
        "revision_count": 0
    }




Critic node:



In [ ]:
def critic_node(state):
    critique = "The answer needs an example."

    return {
        "critique": critique
    }




Revision node:



In [ ]:
def revise_node(state):
    improved_answer = (
        state["draft_answer"]
        + "\nExample: ..."
    )

    return {
        "draft_answer": improved_answer,
        "revision_count": (
            state["revision_count"] + 1
        )
    }




---

# 10. Self-Correction

Self-correction occurs when the agent detects a failure and changes its strategy.

Example:



```text
Action:
Search database using customer ID

Observation:
Customer not found

Correction:
Search using registered email instead
```



Self-correction may be triggered by:

- tool failure,
- empty result,
- invalid output,
- schema validation error,
- contradiction,
- low confidence,
- failed business rule.

---

# 11. Retry Logic

Retries should generally be controlled by normal code rather than unrestricted LLM decisions.

Example:



In [ ]:
MAX_RETRIES = 3


def call_api_with_retry():
    for attempt in range(MAX_RETRIES):
        try:
            return external_api_call()

        except TimeoutError:
            if attempt == MAX_RETRIES - 1:
                raise




An agent should not retry forever.

Always define:

- maximum iterations,
- maximum retries,
- timeout,
- fallback behaviour,
- human escalation condition.

---

# 12. Conditional Architecture

Conditional routing allows the graph to choose different paths.



```text
Check payment status
         ↓
 ┌───────┼────────┐
Success  Pending  Failed
  ↓        ↓        ↓
Confirm  Wait     Refund
```



LangGraph example:



In [ ]:
from typing import Literal


def route_payment(
    state
) -> Literal[
    "confirm",
    "wait",
    "refund"
]:
    status = state["payment_status"]

    if status == "success":
        return "confirm"

    if status == "pending":
        return "wait"

    return "refund"




---

# 13. Sequential Architecture

Nodes execute one after another.



```text
START
  ↓
extract
  ↓
validate
  ↓
process
  ↓
respond
  ↓
END
```



Use this when the order of execution is fixed.

---

# 14. Parallel Architecture

Independent tasks execute in parallel.



```text
             START
            /     \
           ↓       ↓
   search_web   search_database
           \       /
            ↓     ↓
             combine
               ↓
              END
```



Use parallel execution when tasks do not depend on each other.

Examples:

- search multiple data sources,
- retrieve documents from multiple indexes,
- run several evaluations,
- collect pricing from multiple systems.

---

# 15. Supervisor Architecture

A supervisor agent controls multiple specialist agents.



```text
              Supervisor
             /     |     \
            ↓      ↓      ↓
      Research   SQL    Writing
       Agent    Agent    Agent
```



The supervisor decides:

- which specialist should work,
- what task to delegate,
- whether another specialist is needed,
- when the final response is ready.

---

# Example

User:

> Analyse last month’s sales and prepare an executive report.

Supervisor may delegate:



```text
SQL Agent:
Retrieve sales data

Analysis Agent:
Calculate trends and anomalies

Chart Agent:
Create visualisations

Writing Agent:
Prepare executive summary
```



---

# 16. Router Architecture

A router selects one path based on user intent.



```text
User request
     ↓
Intent router
 ┌────┼─────┐
 ↓    ↓     ↓
SQL  RAG  Support
```



Example:



In [ ]:
def route_query(state):
    query_type = state["query_type"]

    if query_type == "database":
        return "sql_agent"

    if query_type == "documents":
        return "rag_agent"

    return "general_agent"




A router usually selects a path once, while a supervisor may repeatedly coordinate multiple workers.

---

# Router vs Supervisor

| Router | Supervisor |
|---|---|
| Selects a path | Coordinates workers |
| Usually one routing decision | May make many decisions |
| Simpler | More flexible |
| Good for intent classification | Good for complex multi-agent work |

---

# 17. Human-in-the-Loop Architecture

Some actions should pause for human approval.



```text
Agent prepares refund
        ↓
Human approval
   ├── Approved → Issue refund
   └── Rejected → Cancel action
```



Use human approval for:

- transferring money,
- issuing large refunds,
- sending sensitive emails,
- deleting data,
- changing permissions,
- executing legal or medical actions,
- publishing important content.

---

# 18. Deterministic Workflow vs Agentic Loop

## Deterministic workflow



```text
Step A → Step B → Step C
```



The path is predefined.

## Agentic loop



```text
Observe
  ↓
Choose next action dynamically
  ↓
Act
  ↓
Observe again
```



Production systems often combine both.

Example:



```text
Deterministic authentication
          ↓
Agentic issue analysis
          ↓
Deterministic approval rule
          ↓
Tool execution
```



---

# 19. Choosing the Right Architecture

Use **ReAct** when:

- the task requires dynamic tool use,
- the next action depends on the last result,
- the task is relatively short.

Use **plan-and-execute** when:

- the task is complex,
- progress tracking is important,
- many subtasks must be completed.

Use **reflection** when:

- quality matters,
- the first answer may be incomplete,
- verification is necessary.

Use **router architecture** when:

- requests belong to clear categories,
- each category has a specialist workflow.

Use **supervisor architecture** when:

- multiple specialist agents must collaborate,
- delegation is dynamic.

Use **human-in-the-loop** when:

- the action is risky,
- approval is legally or operationally required.

---

# 20. Production Safety Controls

Every agent architecture should include stopping conditions.



In [ ]:
MAX_ITERATIONS = 10
MAX_TOOL_CALLS = 8
MAX_RETRIES = 3




Also include:

- tool permissions,
- input validation,
- output validation,
- timeout handling,
- error recovery,
- audit logs,
- cost limits,
- human escalation.

Without stopping conditions, an agent may enter an infinite loop.

---

# Interview Answer

> Common AI-agent architectures include ReAct, plan-and-execute, reflection, routing, supervisor-worker and human-in-the-loop systems. ReAct alternates between reasoning, tool use and observation. Plan-and-execute creates a structured plan before completing individual tasks. Reflection introduces a critic or evaluator that checks and improves the result. Router architectures select a specialist workflow, while supervisors coordinate multiple agents. In production, these architectures are usually combined with deterministic rules, retries, stopping conditions and human approval for high-risk actions.

# Module 3 Summary



```text
ReAct             → Reason, act and observe repeatedly
Plan-and-execute  → Create a plan, then execute its steps
Reflection        → Critique and improve the output
Router            → Select the correct workflow
Supervisor        → Coordinate multiple agents
Human-in-the-loop → Request approval for risky actions
```



The central architectural loop remains:



```text
State → Decision → Action → Observation → State update
```